In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from app.estimate_loader import EstimateLoader

In [2]:
estimates_dir = Path(
    "/Users/himynameisartem/Documents/Development/"
    "Data Science/Renovation_Calculator_CV_Data/estimates"
)

In [3]:
loader = EstimateLoader(estimates_dir)

all_documents = []
errors = []

In [4]:
for estimate_path in sorted(estimates_dir.glob("*.xlsx")):
    try:
        documents = loader.load_workbook_documents(estimate_path)
        all_documents.extend(documents)

        summaries = [
            doc for doc in documents
            if doc.metadata["estimate_document_type"] == "room_summary"
        ]

        groups = [
            doc for doc in documents
            if doc.metadata["estimate_document_type"] == "work_group"
        ]

        line_items = sum(
            doc.metadata["line_items_count"]
            for doc in groups
        )

        print(f"\n{estimate_path.name}")
        print(f"  Помещений/разделов: {len(summaries)}")
        print(f"  Групп работ:        {len(groups)}")
        print(f"  Позиций работ:      {line_items}")

    except Exception as error:
        errors.append((estimate_path.name, str(error)))
        print(f"\nОШИБКА: {estimate_path.name}")
        print(error)

print("\n" + "=" * 50)
print(f"Всего документов: {len(all_documents)}")
print(f"Файлов с ошибками: {len(errors)}")


smeta_117.xlsx
  Помещений/разделов: 2
  Групп работ:        6
  Позиций работ:      40

smeta_269.xlsx
  Помещений/разделов: 1
  Групп работ:        6
  Позиций работ:      15

smeta_279.xlsx
  Помещений/разделов: 9
  Групп работ:        29
  Позиций работ:      168

smeta_280.xlsx
  Помещений/разделов: 10
  Групп работ:        33
  Позиций работ:      190

smeta_281.xlsx
  Помещений/разделов: 12
  Групп работ:        58
  Позиций работ:      302

smeta_287.xlsx
  Помещений/разделов: 9
  Групп работ:        31
  Позиций работ:      158

smeta_289.xlsx
  Помещений/разделов: 7
  Групп работ:        27
  Позиций работ:      160

smeta_307.xlsx
  Помещений/разделов: 11
  Групп работ:        57
  Позиций работ:      346

smeta_310.xlsx
  Помещений/разделов: 12
  Групп работ:        60
  Позиций работ:      455

smeta_362.xlsx
  Помещений/разделов: 7
  Групп работ:        25
  Позиций работ:      122

smeta_37.xlsx
  Помещений/разделов: 10
  Групп работ:        69
  Позиций работ:      273

In [5]:
summary_documents = [
    doc for doc in all_documents
    if doc.metadata["estimate_document_type"] == "room_summary"
]

group_documents = [
    doc for doc in all_documents
    if doc.metadata["estimate_document_type"] == "work_group"
]

total_line_items = sum(
    doc.metadata["line_items_count"]
    for doc in group_documents
)

print("Резюме помещений:", len(summary_documents))
print("Группы работ:", len(group_documents))
print("Позиции работ:", total_line_items)

Резюме помещений: 90
Группы работ: 401
Позиции работ: 2229


In [6]:
from collections import Counter

room_types = Counter(
    doc.metadata["room_type"]
    for doc in summary_documents
)

stage_codes = Counter(
    doc.metadata["stage_code"]
    for doc in group_documents
)

surface_codes = Counter(
    doc.metadata["surface_code"]
    for doc in group_documents
)

print("ТИПЫ ПОМЕЩЕНИЙ")
print(room_types)

print("\nЭТАПЫ")
print(stage_codes)

print("\nПОВЕРХНОСТИ")
print(surface_codes)

ТИПЫ ПОМЕЩЕНИЙ
Counter({'bathroom': 16, 'plumbing': 10, 'general_work': 9, 'electrical': 9, 'kitchen_living_room': 8, 'heating': 8, 'bedroom': 7, 'children_room': 6, 'room': 5, 'hallway': 5, 'wardrobe': 2, 'laundry': 2, 'office': 1, 'balcony': 1, 'kitchen': 1})

ЭТАПЫ
Counter({'rough_finish': 160, 'finish': 146, 'demolition': 53, 'plumbing': 22, 'preparation': 11, 'electrical': 9})

ПОВЕРХНОСТИ
Counter({'walls': 130, 'floor': 118, 'ceiling': 54, 'slopes': 37, 'plumbing': 33, 'other': 11, 'electrical': 10, 'doors_windows': 8})


In [7]:
unknown_rooms = sorted({
    doc.metadata["room_name"]
    for doc in summary_documents
    if doc.metadata["room_type"] == "other"
})

unknown_stages = sorted({
    doc.metadata["stage_name"]
    for doc in group_documents
    if doc.metadata["stage_code"] == "other"
})

print("Неизвестные помещения:", unknown_rooms)
print("Неизвестные этапы:", unknown_stages)

Неизвестные помещения: []
Неизвестные этапы: []


In [8]:
from collections import Counter

work_areas = Counter(
    (
        document.metadata["surface_name"],
        document.metadata["surface_code"],
    )
    for document in group_documents
)

for (name, code), count in sorted(work_areas.items()):
    print(f"{name:<25} → {code:<20} документов: {count}")

Двери, окна               → doors_windows        документов: 3
Двери/окна                → doors_windows        документов: 5
Откосы                    → slopes               документов: 37
Полы                      → floor                документов: 118
Потолки                   → ceiling              документов: 54
Прочее                    → other                документов: 11
Сантехника                → plumbing             документов: 33
Стены                     → walls                документов: 130
Электрика                 → electrical           документов: 10
